# Ensembling — combining already-trained model outputs

The other notebooks train models. This one only **combines predictions that already
exist** — the methodology used for the final leaderboard push, where dozens of
out-of-fold prediction files (our own models + several community-shared OOF libraries)
were blended and stacked without spending extra leaderboard submissions.

To keep this notebook runnable on its own (no cached OOF files needed), it first
trains two small, deliberately different models — LightGBM and a plain Logistic
Regression on standardized features — to get two out-of-fold prediction vectors to
combine. The *combining* methods below are exactly the ones used for real: rank
averaging, OOF weight search, and a logistic-regression stack.

As with the feature-engineering notebook, absolute AUC numbers here are illustrative
(two quick single-seed models), not the production ones.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

import sys, os
sys.path.insert(0, os.getcwd())
from src.config import DATA

SEED = 42
N_SPLITS = 5

train = pd.read_csv(f'{DATA}/train.csv')
y = train['addicted_label'].values

cont_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'notifications_per_day',
             'app_opens_per_day', 'weekend_screen_time']
X_raw = train[cont_cols].apply(pd.to_numeric, errors='coerce')

## 1) Two out-of-fold prediction sources

Same 5-fold split for both, so the OOF vectors are directly comparable — this is the
same discipline the real project used when comparing/blending model families.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(y))
oof_lr = np.zeros(len(y))

X_lr = X_raw.fillna(X_raw.median())
scaler = StandardScaler().fit(X_lr)
X_lr_scaled = pd.DataFrame(scaler.transform(X_lr), columns=X_lr.columns, index=X_lr.index)

for tr_idx, va_idx in skf.split(X_raw, y):
    gbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                              random_state=SEED, verbosity=-1)
    gbm.fit(X_raw.iloc[tr_idx], y[tr_idx])
    oof_lgb[va_idx] = gbm.predict_proba(X_raw.iloc[va_idx])[:, 1]

    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X_lr_scaled.iloc[tr_idx], y[tr_idx])
    oof_lr[va_idx] = lr.predict_proba(X_lr_scaled.iloc[va_idx])[:, 1]

auc_lgb = roc_auc_score(y, oof_lgb)
auc_lr = roc_auc_score(y, oof_lr)
corr = np.corrcoef(oof_lgb, oof_lr)[0, 1]
print(f'LightGBM solo AUC:  {auc_lgb:.5f}')
print(f'LogReg solo AUC:    {auc_lr:.5f}')
print(f'Pearson correlation between the two: {corr:.4f}')

## 2) Rank-average blend

The simplest combination method, and the one used for every GBDT/XGBoost/CatBoost
blend in the production pipeline: convert each model's predictions to ranks (so scale
differences between models don't matter), then average.

In [ ]:
def rank_average(*preds):
    ranks = [rankdata(p) for p in preds]
    return np.mean(ranks, axis=0) / len(preds[0])

oof_blend_rank = rank_average(oof_lgb, oof_lr)
auc_blend_rank = roc_auc_score(y, oof_blend_rank)
print(f'Rank-average blend AUC: {auc_blend_rank:.5f}  (best solo: {max(auc_lgb, auc_lr):.5f})')

## 3) OOF weight search

Rather than a fixed 50/50 blend, scan blend weights against the out-of-fold AUC and
pick the optimum — this is exactly how the GBDT/NN blend weight was chosen in the real
pipeline (see `src/ensembling/blend_gbdt_origfeat_nn_featfull_2026-08-29.py`), which
avoids spending a leaderboard submission per weight tried.

In [ ]:
weights = np.linspace(0, 1, 101)
aucs = [roc_auc_score(y, w * oof_lgb + (1 - w) * oof_lr) for w in weights]
best_w = weights[int(np.argmax(aucs))]
print(f'Best weight on LightGBM: {best_w:.2f}  ->  OOF AUC {max(aucs):.5f}')

## 4) Logistic-regression stack

A second-level model that learns how to combine the base predictions, instead of a
fixed weight — the same idea behind the final leaderboard push's ~100-member OOF-library
stack, just with two members here instead of ~100.

In [ ]:
stack_X = np.column_stack([oof_lgb, oof_lr])
stack_oof = np.zeros(len(y))
for tr_idx, va_idx in skf.split(stack_X, y):
    meta = LogisticRegression()
    meta.fit(stack_X[tr_idx], y[tr_idx])
    stack_oof[va_idx] = meta.predict_proba(stack_X[va_idx])[:, 1]

auc_stack = roc_auc_score(y, stack_oof)
print(f'Logistic-regression stack AUC: {auc_stack:.5f}')

## Summary

| Method | AUC |
|---|---|
| LightGBM solo | see output |
| Logistic Regression solo | see output |
| Rank-average blend | see output |
| Weight-searched blend | see output |
| Logistic-regression stack | see output |

The real project repeated this exact logic at much larger scale: dozens of OOF members
pooled from several community-shared libraries, a 2nd-level meta-model over several
1st-level combination strategies, and a repeatedly-validated rule for when a small OOF
delta (as little as 0.00005–0.00017 AUC here) is real signal versus noise. See the main
README's "Key findings" and `src/ensembling/` for the full version.